# Bài 21: Rút mã cổ phiếu từ câu hỏi
## Bỏ ô 'Mã cổ phiếu' — user chỉ cần hỏi tự nhiên

**Mục tiêu:**
- Thêm node `symbol_extraction` chạy TRƯỚC data_collection
- Dùng LLM đọc câu hỏi → trả về list mã `["NVDA", "TSLA"]`
- Bỏ ô nhập mã ở sidebar → app thành chat thật sự

**Kết nối:** đây chính là LLM router bài 14, nhưng giờ đặt vào pipeline product. Thay vì user khai mã 2 nơi (ô sidebar + câu hỏi), hệ thống tự rút mã từ câu hỏi.

---
## Phần 1: Lý thuyết

### Vấn đề hiện tại

```
Sidebar: [Mã cổ phiếu: NVDA,AMD]   ← user gõ ở đây
Chat:    "So sánh NVDA và AMD"     ← rồi nhắc lại ở đây
```

Khai 2 nơi → thừa và clunky. Product chat (ChatGPT, Perplexity) chỉ cần bạn hỏi, nó tự hiểu công ty nào.

---

### Giải pháp: một node LLM rút mã

Thêm một bước vào graph, chạy đầu tiên:

```
START → symbol_extraction → data_collection → analysis → END
         (MỚI: đọc câu hỏi     (dùng symbols     (tổng hợp)
          → state["symbols"])    node trên tạo)
```

Node này nhận câu hỏi, gọi LLM trả JSON, ghi kết quả vào `state["symbols"]`. Các node sau dùng y như cũ — **không cần sửa** data_collection hay analysis.

---

### LLM trả JSON — nhắc lại bài 14

```python
prompt = 'Trả về JSON {"symbols": [...]}. Câu hỏi: So sánh Nvidia và Tesla'
# LLM trả: {"symbols": ["NVDA", "TSLA"]}
```

Điểm mạnh so với tách chữ thủ công: LLM hiểu *"Nvidia" → NVDA*, *"Tesla" → TSLA*, *"hãng táo" → AAPL*. Nó ánh xạ tên công ty → ticker chuẩn.

---

### Dùng lịch sử để hiểu câu hỏi nối tiếp

Câu follow-up như *"còn AMD thì sao?"* chỉ có nghĩa khi biết ngữ cảnh trước. Nên ta đưa **cả lịch sử hội thoại** vào prompt để LLM hiểu:

```
Lịch sử: [user hỏi so sánh NVDA và TSLA...]
Câu hiện tại: "còn AMD thì sao?"
→ LLM có thể trả ["AMD"] hoặc ["NVDA", "TSLA", "AMD"] tùy ngữ cảnh
```

---

### Fallback khi không có mã

Câu hỏi kiểu *"P/E là gì?"* không nhắc công ty nào → trả `[]`. data_collection lặp qua list rỗng → không lấy dữ liệu công ty → analysis vẫn trả lời khái niệm bình thường. Linh hoạt hơn hẳn bản cũ (bắt buộc phải có mã).

---
## Phần 2: Ví dụ — thử rút mã bằng Gemini

Chạy ô dưới để thấy LLM ánh xạ tên công ty → ticker.

In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

EXTRACT_PROMPT = """Bạn là bộ trích xuất mã cổ phiếu (ticker).
Từ câu hỏi, xác định mã ticker chuẩn sàn Mỹ của các công ty được nhắc tới.
- Trả về DUY NHẤT JSON: {{"symbols": ["NVDA", "TSLA"]}}
- Ánh xạ tên → ticker: Nvidia→NVDA, Tesla→TSLA, Apple→AAPL, Microsoft→MSFT...
- Không nhắc công ty nào → {{"symbols": []}}

Câu hỏi: {question}
"""

def extract_symbols(question: str) -> list[str]:
    prompt = EXTRACT_PROMPT.format(question=question)
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    text = response.text.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])   # bỏ ```json ... ```
    return json.loads(text).get("symbols", [])

for q in [
    "So sánh Nvidia và Tesla quý gần nhất",
    "Doanh thu của hãng táo thế nào?",
    "P/E ratio là gì?",
]:
    print(f"{q!r}\n  → {extract_symbols(q)}\n")

'So sánh Nvidia và Tesla quý gần nhất'
  → ['NVDA', 'TSLA']

'Doanh thu của hãng táo thế nào?'
  → ['AAPL']

'P/E ratio là gì?'
  → []



Quan sát: *"Nvidia và Tesla"* → `["NVDA", "TSLA"]`, *"hãng táo"* → `["AAPL"]`, câu khái niệm → `[]`. Đây là thứ ta cần.

---
## Phần 3: Bài tập

### Bước 1: Tạo `app/agents/symbol_extraction.py`

Khung có sẵn — điền `TODO`. Khác ví dụ Phần 2 ở chỗ: có thêm **lịch sử** và **try/except fallback**.

```python
import json
from config import client, MODEL_NAME
from logging_setup import logger
from core.state import AppState

EXTRACT_PROMPT = """Bạn là bộ trích xuất mã cổ phiếu (ticker).
Từ câu hỏi (và lịch sử nếu cần), xác định mã ticker chuẩn sàn Mỹ.
- Trả về DUY NHẤT JSON: {{"symbols": ["NVDA", "TSLA"]}}
- Ánh xạ tên → ticker: Nvidia→NVDA, Tesla→TSLA, Apple→AAPL...
- Không nhắc công ty nào → {{"symbols": []}}

Lịch sử hội thoại:
{history}

Câu hỏi hiện tại: {question}
"""

def symbol_extraction_node(state: AppState) -> dict:
    question = state["messages"][-1]["content"]

    # TODO: dựng history từ state["messages"][:-1] (giống analysis_node)
    #   mỗi dòng: "Người dùng: ..." / "Assistant: ..."
    history = ""
    for msg in state["messages"][:-1]:
        role = "Người dùng" if msg["role"] == "user" else "Assistant"
        history += f"{role}: {msg['content']}\n"

    prompt = EXTRACT_PROMPT.format(history=history, question=question)

    try:
        response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
        text = response.text.strip()
        if text.startswith("```"):
            text = "\n".join(text.split("\n")[1:-1])
        # TODO: parse JSON → lấy list symbols
        symbols = json.loads(text).get("symbols", [])
    except Exception as e:
        logger.error(f"[symbol_extraction] lỗi parse: {e} → fallback []")
        symbols = []

    logger.info(f"[symbol_extraction] rút được: {symbols}")
    return {"symbols": symbols}
```

---

### Bước 2: Nối node vào graph — `app/core/graph.py`

Thêm node mới vào ĐẦU pipeline. Sửa (điền TODO):

```python
from agents.symbol_extraction import symbol_extraction_node   # ← import mới

def get_graph():
    builder = StateGraph(AppState)
    builder.add_node("symbol_extraction", symbol_extraction_node)   # ← node mới
    builder.add_node("data_collection", data_collection_node)
    builder.add_node("analysis", analysis_node)

    # TODO: sửa lại các edge cho đúng thứ tự:
    #   START → symbol_extraction → data_collection → analysis → END
    builder.add_edge(START, "symbol_extraction")
    builder.add_edge("symbol_extraction", "data_collection")
    builder.add_edge("data_collection", "analysis")
    builder.add_edge("analysis", END)

    memory = MemorySaver()
    return builder.compile(checkpointer=memory)
```

---

### Bước 3: Bỏ ô 'Mã cổ phiếu' ở sidebar — `app/main.py`

**3a.** Xóa khối nhập mã trong `with st.sidebar:` (đoạn `symbols_input = st.text_input(...)` và dòng `symbols = [...]`).

**3b.** Trong `initial_state`, `symbols` giờ do node tự điền → khởi tạo rỗng:
```python
        initial_state: AppState = {
            "symbols": [],          # ← node symbol_extraction sẽ ghi đè
            "messages": [{"role": "user", "content": question}],
            "company_data": {},
            "sources": [],
        }
```

> Lưu ý: biến `symbols` cũ không còn được dùng trong `initial_state` nữa — nếu còn chỗ nào tham chiếu `symbols` (ngoài đây) thì xóa luôn.

---

### Bước 4: Test

`streamlit run app/main.py` — sidebar giờ **không còn ô Mã cổ phiếu**. Thử:

1. Hỏi "So sánh Nvidia và Tesla" → log phải hiện `[symbol_extraction] rút được: ['NVDA', 'TSLA']`, rồi data_collection chạy cho đúng 2 mã đó
2. Hỏi tiếp "còn AMD thì sao?" → xem node có hiểu ngữ cảnh không
3. Hỏi "P/E ratio là gì?" → rút `[]`, vẫn trả lời khái niệm

**Checklist:**
- [ ] Sidebar sạch, không còn ô nhập mã
- [ ] Hỏi tự nhiên (không gõ ticker in hoa) vẫn ra đúng công ty
- [ ] Câu hỏi không có công ty → không crash, trả lời chung